In [0]:
from pyspark.sql import functions as F

# Tabela fato
fato = spark.table("workspace.gold.fato_acidentes")

# Dimensões
dim_tempo = spark.table("workspace.gold.dim_tempo")
dim_horario = spark.table("workspace.gold.dim_horario")
dim_bairro = spark.table("workspace.gold.dim_bairro")
dim_regiao = spark.table("workspace.gold.dim_regiao")

print("Tabelas Gold carregadas com sucesso.")

Tabelas Gold carregadas com sucesso.


Quais bairros apresentam maior número de acidentes de trânsito?

In [0]:
# Pergunta 1:
# Quais bairros apresentam maior número de acidentes de trânsito?

acidentes_por_bairro = (
    fato.alias("f")
    .join(
        dim_bairro.alias("b"),
        F.col("f.id_bairro") == F.col("b.id_bairro"),
        "inner"
    )
    .filter(F.col("f.id_bairro") != -1)
    .groupBy(
        F.col("b.bairro")
    )
    .agg(
        F.count("*").alias("total_acidentes")
    )
    .orderBy(
        F.desc("total_acidentes"),
        F.asc("bairro")
    )
)

display(
    acidentes_por_bairro.limit(20)
)

bairro,total_acidentes
CAMPO GRANDE,1618
BARRA DA TIJUCA,1360
BANGU,1011
SANTA CRUZ,809
CENTRO,797
BONSUCESSO,776
RECREIO DOS BANDEIRANTES,772
GUARATIBA,750
REALENGO,725
TAQUARA,549


In [0]:
# Cobertura da análise por bairro

total_acidentes = fato.count()

acidentes_com_bairro = (
    fato
    .filter(F.col("id_bairro") != -1)
    .count()
)

acidentes_sem_bairro = total_acidentes - acidentes_com_bairro

cobertura_bairro = (
    acidentes_com_bairro / total_acidentes * 100
)

print(f"Total de acidentes: {total_acidentes}")
print(f"Acidentes com bairro identificado: {acidentes_com_bairro}")
print(f"Acidentes sem bairro identificado: {acidentes_sem_bairro}")
print(f"Cobertura da análise por bairro: {cobertura_bairro:.2f}%")

# Participação de cada bairro entre os acidentes com bairro identificado
ranking_bairros = (
    acidentes_por_bairro
    .withColumn(
        "participacao_percentual",
        F.round(
            F.col("total_acidentes")
            / F.lit(acidentes_com_bairro) * 100,
            2
        )
    )
)

display(
    ranking_bairros.limit(20)
)

Total de acidentes: 55046
Acidentes com bairro identificado: 28152
Acidentes sem bairro identificado: 26894
Cobertura da análise por bairro: 51.14%


bairro,total_acidentes,participacao_percentual
CAMPO GRANDE,1618,5.75
BARRA DA TIJUCA,1360,4.83
BANGU,1011,3.59
SANTA CRUZ,809,2.87
CENTRO,797,2.83
BONSUCESSO,776,2.76
RECREIO DOS BANDEIRANTES,772,2.74
GUARATIBA,750,2.66
REALENGO,725,2.58
TAQUARA,549,1.95


Existe sazonalidade na ocorrência de acidentes ao longo dos meses?


In [0]:
# Pergunta 2:
# Existe sazonalidade na ocorrência de acidentes ao longo dos meses?

acidentes_mes_ano = (
    fato.alias("f")
    .join(
        dim_tempo.alias("t"),
        F.col("f.id_tempo") == F.col("t.id_tempo"),
        "inner"
    )
    .groupBy(
        F.col("t.ano"),
        F.col("t.mes")
    )
    .agg(
        F.count("*").alias("total_acidentes")
    )
)

sazonalidade_mensal = (
    acidentes_mes_ano
    .groupBy("mes")
    .agg(
        F.round(
            F.avg("total_acidentes"),
            2
        ).alias("media_acidentes_mes"),

        F.min("total_acidentes").alias("minimo"),

        F.max("total_acidentes").alias("maximo"),

        F.countDistinct("ano").alias("anos_disponiveis")
    )
    .orderBy("mes")
)

display(sazonalidade_mensal)

mes,media_acidentes_mes,minimo,maximo,anos_disponiveis
1,666.0,554,820,7
2,636.71,514,788,7
3,687.71,448,900,7
4,628.83,259,870,6
5,637.5,272,805,6
6,685.0,360,827,7
7,673.43,492,797,7
8,746.0,565,885,6
9,699.14,426,877,7
10,732.86,577,861,7


In [0]:
# Detalhamento mensal por ano para avaliar a sazonalidade

matriz_mensal = (
    acidentes_mes_ano
    .groupBy("ano")
    .pivot("mes", list(range(1, 13)))
    .agg(F.first("total_acidentes"))
    .orderBy("ano")
)

display(matriz_mensal)

ano,1,2,3,4,5,6,7,8,9,10,11,12
2018,691,671,900,870,805,824,797,834,803,816,772,900
2019,740,701,776,744,801,827,774,885,814,861,753,773
2020,628,620,448,259,272,360,535,565,560,577,695,517
2021,634,573,624,550,557,574,625,650,426,587,532,586
2022,554,514,728,598,615,686,492,739,655,686,623,729
2023,595,590,782,752,null,758,772,null,877,809,995,869
2024,820,788,556,null,775,766,719,803,759,794,762,null


In [0]:
# Sazonalidade mensal utilizando somente anos completos
# Período completo identificado: 2018 a 2022

sazonalidade_anos_completos = (
    acidentes_mes_ano
    .filter(F.col("ano").between(2018, 2022))
    .groupBy("mes")
    .agg(
        F.round(
            F.avg("total_acidentes"),
            2
        ).alias("media_mensal"),
        
        F.sum("total_acidentes").alias("total_periodo"),
        
        F.min("total_acidentes").alias("minimo"),
        
        F.max("total_acidentes").alias("maximo")
    )
    .orderBy("mes")
)

display(sazonalidade_anos_completos)

mes,media_mensal,total_periodo,minimo,maximo
1,649.4,3247,554,740
2,615.8,3079,514,701
3,695.2,3476,448,900
4,604.2,3021,259,870
5,610.0,3050,272,805
6,654.2,3271,360,827
7,644.6,3223,492,797
8,734.6,3673,565,885
9,651.6,3258,426,814
10,705.4,3527,577,861


Quais dias da semana apresentam maior frequência de acidentes?

In [0]:
# Pergunta 3:
# Quais dias da semana apresentam maior frequência de acidentes?

acidentes_dia_semana = (
    fato.alias("f")
    .join(
        dim_tempo.alias("t"),
        F.col("f.id_tempo") == F.col("t.id_tempo"),
        "inner"
    )
    .groupBy(
        F.col("t.dia_semana_num"),
        F.col("t.dia_semana")
    )
    .agg(
        F.count("*").alias("total_acidentes")
    )
    .withColumn(
        "participacao_percentual",
        F.round(
            F.col("total_acidentes")
            / F.lit(fato.count()) * 100,
            2
        )
    )
    .orderBy(F.desc("total_acidentes"))
)

display(acidentes_dia_semana)

dia_semana_num,dia_semana,total_acidentes,participacao_percentual
2,SEGUNDA-FEIRA,8248,14.98
5,QUINTA-FEIRA,8211,14.92
6,SEXTA-FEIRA,8181,14.86
4,QUARTA-FEIRA,7795,14.16
3,TERCA-FEIRA,7591,13.79
1,DOMINGO,7552,13.72
7,SABADO,7468,13.57


Quais horários concentram mais ocorrências?

In [0]:
# Pergunta 4:
# Quais horários concentram maior frequência de acidentes?

acidentes_por_hora = (
    fato.alias("f")
    .join(
        dim_horario.alias("h"),
        F.col("f.id_horario") == F.col("h.id_horario"),
        "inner"
    )
    .groupBy(
        F.col("h.hora")
    )
    .agg(
        F.count("*").alias("total_acidentes")
    )
    .withColumn(
        "participacao_percentual",
        F.round(
            F.col("total_acidentes")
            / F.lit(55046) * 100,
            2
        )
    )
    .orderBy("hora")
)

display(acidentes_por_hora)

hora,total_acidentes,participacao_percentual
0,1518,2.76
1,1010,1.83
2,766,1.39
3,665,1.21
4,812,1.48
5,1305,2.37
6,2097,3.81
7,2753,5.0
8,2809,5.1
9,2660,4.83


In [0]:
# Distribuição dos acidentes por faixa horária

acidentes_faixa_horaria = (
    fato.alias("f")
    .join(
        dim_horario.alias("h"),
        F.col("f.id_horario") == F.col("h.id_horario"),
        "inner"
    )
    .groupBy(
        F.col("h.faixa_horaria")
    )
    .agg(
        F.count("*").alias("total_acidentes")
    )
    .withColumn(
        "participacao_percentual",
        F.round(
            F.col("total_acidentes")
            / F.lit(55046) * 100,
            2
        )
    )
    .orderBy(F.desc("total_acidentes"))
)

display(acidentes_faixa_horaria)

# Pico horário exato
print("Horas com maior número de acidentes:")

display(
    acidentes_por_hora
    .orderBy(F.desc("total_acidentes"))
    .limit(10)
)

faixa_horaria,total_acidentes,participacao_percentual
TARDE,17213,31.27
NOITE,15910,28.9
MANHA,15847,28.79
MADRUGADA,6076,11.04


Horas com maior número de acidentes:


hora,total_acidentes,participacao_percentual
19,3349,6.08
18,3348,6.08
17,3145,5.71
15,2952,5.36
16,2866,5.21
8,2809,5.1
12,2798,5.08
10,2776,5.04
20,2769,5.03
7,2753,5.0


Existem diferenças na ocorrência de acidentes entre as Regiões Administrativas?

In [0]:
# Pergunta 5:
# Como os acidentes se distribuem entre as Regiões Administrativas?

total_com_ra = (
    fato
    .filter(F.col("id_regiao") != -1)
    .count()
)

acidentes_por_regiao = (
    fato.alias("f")
    .join(
        dim_regiao.alias("r"),
        F.col("f.id_regiao") == F.col("r.id_regiao"),
        "inner"
    )
    .filter(F.col("f.id_regiao") != -1)
    .groupBy(
        F.col("r.id_regiao"),
        F.col("r.regiao_administrativa"),
        F.col("r.area_planejamento")
    )
    .agg(
        F.count("*").alias("total_acidentes")
    )
    .withColumn(
        "participacao_percentual",
        F.round(
            F.col("total_acidentes")
            / F.lit(total_com_ra) * 100,
            2
        )
    )
    .orderBy(F.desc("total_acidentes"))
)

print(f"Acidentes com RA determinada: {total_com_ra}")
print(
    f"Cobertura da análise regional: "
    f"{(total_com_ra / 55046) * 100:.2f}%"
)

display(
    acidentes_por_regiao.limit(20)
)

Acidentes com RA determinada: 27075
Cobertura da análise regional: 49.19%


id_regiao,regiao_administrativa,area_planejamento,total_acidentes,participacao_percentual
24,BARRA DA TIJUCA,4,2397,8.85
18,CAMPO GRANDE,5,2259,8.34
16,JACAREPAGUA,4,2055,7.59
13,MEIER,3,1798,6.64
15,MADUREIRA,3,1560,5.76
10,RAMOS,3,1356,5.01
33,REALENGO,5,1316,4.86
17,BANGU,5,1314,4.85
4,BOTAFOGO,2,1165,4.3
19,SANTA CRUZ,5,1149,4.24


Como evoluiu o número de acidentes ao longo dos anos?

In [0]:
# Pergunta 6:
# Como evoluiu o número de acidentes ao longo dos anos?

evolucao_anual = (
    fato.alias("f")
    .join(
        dim_tempo.alias("t"),
        F.col("f.id_tempo") == F.col("t.id_tempo"),
        "inner"
    )
    .groupBy(
        F.col("t.ano")
    )
    .agg(
        F.count("*").alias("total_acidentes"),
        F.countDistinct("t.mes").alias("meses_disponiveis")
    )
    .orderBy("ano")
)

display(evolucao_anual)

ano,total_acidentes,meses_disponiveis
2018,9683,12
2019,9449,12
2020,6036,12
2021,6918,12
2022,7619,12
2023,7799,10
2024,7542,10


In [0]:
# Evolução anual considerando a cobertura temporal disponível

evolucao_anual_ajustada = (
    evolucao_anual
    .withColumn(
        "media_mensal",
        F.round(
            F.col("total_acidentes")
            / F.col("meses_disponiveis"),
            2
        )
    )
    .withColumn(
        "ano_completo",
        F.when(
            F.col("meses_disponiveis") == 12,
            F.lit("SIM")
        ).otherwise(F.lit("NAO"))
    )
)

display(evolucao_anual_ajustada)

ano,total_acidentes,meses_disponiveis,media_mensal,ano_completo
2018,9683,12,806.92,SIM
2019,9449,12,787.42,SIM
2020,6036,12,503.0,SIM
2021,6918,12,576.5,SIM
2022,7619,12,634.92,SIM
2023,7799,10,779.9,NAO
2024,7542,10,754.2,NAO


Quais problemas de qualidade foram encontrados e qual seria o impacto caso não fossem tratados?

In [0]:
# Pergunta 7:
# Quais problemas de qualidade foram encontrados e qual seria
# o impacto caso não fossem tratados?

resumo_impacto_qualidade = [
    (
        "Ausência de bairro",
        "26.894 acidentes (48,86%)",
        "Reduz a cobertura das análises por bairro e Região Administrativa."
    ),
    (
        "Baixa completude territorial em 2018-2020",
        "0,00% em 2018; 0,01% em 2019; 0,25% em 2020",
        "Impede comparação territorial representativa desses anos com 2021-2024."
    ),
    (
        "Bairro não associado ao cadastro oficial",
        "1.077 acidentes (1,96%)",
        "Pode produzir classificação incorreta de Região Administrativa se houver correção automática."
    ),
    (
        "Grafias e valores não padronizados de bairro",
        "Ex.: OSWALDO CRUZ, 00000000 e BAIRRO NAO CADASTRADO",
        "Pode fragmentar bairros, criar categorias inválidas e distorcer rankings."
    ),
    (
        "Cobertura mensal incompleta",
        "2023 e 2024 possuem 10 meses registrados",
        "Totais anuais desses anos não são diretamente comparáveis aos anos completos."
    )
]

df_impacto_qualidade = spark.createDataFrame(
    resumo_impacto_qualidade,
    [
        "problema",
        "evidencia",
        "impacto_analitico"
    ]
)

display(df_impacto_qualidade)

problema,evidencia,impacto_analitico
Ausência de bairro,"26.894 acidentes (48,86%)",Reduz a cobertura das análises por bairro e Região Administrativa.
Baixa completude territorial em 2018-2020,"0,00% em 2018; 0,01% em 2019; 0,25% em 2020",Impede comparação territorial representativa desses anos com 2021-2024.
Bairro não associado ao cadastro oficial,"1.077 acidentes (1,96%)",Pode produzir classificação incorreta de Região Administrativa se houver correção automática.
Grafias e valores não padronizados de bairro,"Ex.: OSWALDO CRUZ, 00000000 e BAIRRO NAO CADASTRADO","Pode fragmentar bairros, criar categorias inválidas e distorcer rankings."
Cobertura mensal incompleta,2023 e 2024 possuem 10 meses registrados,Totais anuais desses anos não são diretamente comparáveis aos anos completos.


In [0]:
# Schemas físicos das tabelas Gold para documentação do catálogo

tabelas_catalogo = [
    "workspace.gold.dim_tempo",
    "workspace.gold.dim_horario",
    "workspace.gold.dim_bairro",
    "workspace.gold.dim_regiao",
    "workspace.gold.fato_acidentes"
]

for tabela in tabelas_catalogo:
    print("\n" + "=" * 80)
    print(tabela)
    print("=" * 80)
    spark.table(tabela).printSchema()


workspace.gold.dim_tempo
root
 |-- data_acidente: date (nullable = true)
 |-- id_tempo: integer (nullable = true)
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- dia: integer (nullable = true)
 |-- dia_semana_num: integer (nullable = true)
 |-- dia_semana: string (nullable = true)
 |-- trimestre: integer (nullable = true)


workspace.gold.dim_horario
root
 |-- id_horario: integer (nullable = true)
 |-- horario: string (nullable = true)
 |-- hora: integer (nullable = true)
 |-- minuto: integer (nullable = true)
 |-- segundo: integer (nullable = true)
 |-- faixa_horaria: string (nullable = true)


workspace.gold.dim_bairro
root
 |-- id_bairro: long (nullable = true)
 |-- bairro: string (nullable = true)


workspace.gold.dim_regiao
root
 |-- id_regiao: integer (nullable = true)
 |-- regiao_administrativa: string (nullable = true)
 |-- area_planejamento: integer (nullable = true)


workspace.gold.fato_acidentes
root
 |-- num_acidente: string (nullable = true)
